# Spotify Music Recommendation System
## Notebook 01 — Data Understanding

**Purpose:** Load and inspect every raw dataset, understand each column, and expose all hidden data quality issues before any analysis begins.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Datasets](#2-load-datasets)
3. [Dataset Overview](#3-dataset-overview)
4. [Data Dictionary](#4-data-dictionary)
5. [Column-by-Column Inspection](#5-column-by-column-inspection)
6. [True Data Quality Check](#6-true-data-quality-check)
7. [Conclusion](#7-conclusion)

## 1. Setup & Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data/raw')

## 2. Load Datasets

In [2]:
data       = pd.read_csv(DATA_DIR / 'data.csv')
data_genre = pd.read_csv(DATA_DIR / 'data_w_genres.csv')
artist_df  = pd.read_csv(DATA_DIR / 'data_by_artist.csv')
genre_df   = pd.read_csv(DATA_DIR / 'data_by_genres.csv')
year_df    = pd.read_csv(DATA_DIR / 'data_by_year.csv')
song_df    = pd.read_csv(DATA_DIR / 'song_and_artists.csv')

datasets = {
    'data'            : data,
    'data_w_genres'   : data_genre,
    'data_by_artist'  : artist_df,
    'data_by_genres'  : genre_df,
    'data_by_year'    : year_df,
    'song_and_artists': song_df,
}
print('All datasets loaded.')

All datasets loaded.


## 3. Dataset Overview

In [3]:
summary = pd.DataFrame({
    'Rows'    : [df.shape[0] for df in datasets.values()],
    'Columns' : [df.shape[1] for df in datasets.values()],
    'Mem (MB)': [round(df.memory_usage(deep=True).sum()/1e6, 2) for df in datasets.values()],
    'Null (pandas)': [df.isnull().sum().sum() for df in datasets.values()],
}, index=datasets.keys())
summary

,Rows,Columns,Mem (MB),Null (pandas)
data,170653,19,73.410,0
data_w_genres,28680,16,7.930,0
data_by_artist,28680,15,5.280,0
data_by_genres,2973,14,0.520,0
data_by_year,100,14,0.010,0
song_and_artists,2420,5,0.870,13


## 4. Data Dictionary

### Spotify Audio Features (all in `data.csv` and aggregated datasets)

| Column | Type | Range | What it measures |
|---|---|---|---|
| `valence` | float | 0–1 | Musical happiness (1 = joyful, 0 = sad) |
| `acousticness` | float | 0–1 | How acoustic the track is (1 = fully acoustic) |
| `danceability` | float | 0–1 | How suitable for dancing (rhythm, beat strength) |
| `energy` | float | 0–1 | Intensity and activity (loud, fast = high) |
| `instrumentalness` | float | 0–1 | Absence of vocals (>0.5 = likely no singing) |
| `liveness` | float | 0–1 | Presence of live audience (>0.8 = live recording) |
| `loudness` | float | −60–0 dB | Overall loudness in decibels |
| `speechiness` | float | 0–1 | Spoken words (>0.66 = speech, <0.33 = music) |
| `tempo` | float | BPM | Speed in beats per minute |
| `key` | int | 0–11 | Musical key (0=C, 1=C#, …, 11=B) |
| `mode` | int | 0 or 1 | 1=major key, 0=minor key |

### Track Metadata

| Column | Type | Notes |
|---|---|---|
| `id` | string | Unique Spotify track ID |
| `name` | string | Track title |
| `artists` | string | **Stored as a Python list string** e.g. `"['Artist A', 'Artist B']"` |
| `year` | int | Release year (clean, always present) |
| `release_date` | string | **Mixed format** — `'YYYY'`, `'YYYY-MM-DD'`, or other |
| `popularity` | int | Spotify score 0–100 (recency-weighted) |
| `explicit` | int | 1 = explicit lyrics, 0 = clean |
| `duration_ms` | int | Track length in milliseconds |
| `genres` | string | In `data_w_genres` — **list string**, `'[]'` means no genre |
| `count` | int | Number of tracks averaged (in artist/genre datasets) |

## 5. Column-by-Column Inspection

### 5.1 `data.csv` — Main Track Dataset

In [4]:
print(f'Shape: {data.shape}')
print(f'Years covered: {data["year"].min()} – {data["year"].max()}')
print(f'Unique tracks (by id): {data["id"].nunique():,}')
print(f'Unique song names    : {data["name"].nunique():,}  (lower = some names repeat as covers)')
data.head(3)

Shape: (170653, 19)
Years covered: 1921 – 2020
Unique tracks (by id): 170,653
Unique song names    : 133,638  (lower = some names repeat as covers)


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.059,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.037,80.954
1,0.963,1921,0.732,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.415,60.936
2,0.039,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913,3,0.101,-14.850,1,Gati Bali,5,1921,0.034,110.339


In [5]:
# Show that artists is a Python list stored as a string
print('artists column — raw values:')
for val in data['artists'].head(4):
    print(' ', repr(val))
print()
print('release_date — value format variety:')
year_only  = data['release_date'].str.match(r'^\d{4}$').sum()
full_date  = data['release_date'].str.match(r'^\d{4}-\d{2}-\d{2}$').sum()
other_fmt  = len(data) - year_only - full_date
print(f'  Year only  (YYYY)        : {year_only:,}')
print(f'  Full date  (YYYY-MM-DD)  : {full_date:,}')
print(f'  Other formats            : {other_fmt:,}')

artists column — raw values:
  "['Sergei Rachmaninoff', 'James Levine', 'Berliner Philharmoniker']"
  "['Dennis Day']"
  "['KHP Kridhamardawa Karaton Ngayogyakarta Hadiningrat']"
  "['Frank Parker']"

release_date — value format variety:
  Year only  (YYYY)        : 50,855
  Full date  (YYYY-MM-DD)  : 118,188
  Other formats            : 1,610


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170653 entries, 0 to 170652
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           170653 non-null  float64
 1   year              170653 non-null  int64  
 2   acousticness      170653 non-null  float64
 3   artists           170653 non-null  object 
 4   danceability      170653 non-null  float64
 5   duration_ms       170653 non-null  int64  
 6   energy            170653 non-null  float64
 7   explicit          170653 non-null  int64  
 8   id                170653 non-null  object 
 9   instrumentalness  170653 non-null  float64
 10  key               170653 non-null  int64  
 11  liveness          170653 non-null  float64
 12  loudness          170653 non-null  float64
 13  mode              170653 non-null  int64  
 14  name              170653 non-null  object 
 15  popularity        170653 non-null  int64  
 16  release_date      17

### 5.2 `data_w_genres.csv` — Artist-Level Data with Genres

In [7]:
print(f'Shape: {data_genre.shape}')
print()
print('genres column — raw values (first 6):')
for val in data_genre['genres'].head(6):
    print(' ', repr(val))
print()
print('artists column — raw values (first 3):')
for val in data_genre['artists'].head(3):
    print(' ', repr(val))
print()
print('NOTE: artists here are plain strings, NOT list strings like in data.csv')

Shape: (28680, 16)

genres column — raw values (first 6):
  "['show tunes']"
  '[]'
  '[]'
  '[]'
  '[]'
  '[]'

artists column — raw values (first 3):
  '"Cats" 1981 Original London Cast'
  '"Cats" 1983 Broadway Cast'
  '"Fiddler On The Roof” Motion Picture Chorus'

NOTE: artists here are plain strings, NOT list strings like in data.csv


### 5.3 `data_by_year.csv` — Yearly Averages

In [8]:
print(f'Shape: {year_df.shape}')
print()
print('mode column unique values:', year_df['mode'].unique())
print('=> mode is CONSTANT (always 1). Zero variance. Useless for analysis.')
print()
print('popularity range (these are MEAN values per year, not individual scores):')
print(year_df[['year', 'popularity']].describe().to_string())
print('\nOld tracks get near-zero popularity because Spotify weights recent plays.')

Shape: (100, 14)

mode column unique values: [1]
=> mode is CONSTANT (always 1). Zero variance. Useless for analysis.

popularity range (these are MEAN values per year, not individual scores):
          year  popularity
count  100.000     100.000
mean  1970.500      27.376
std     29.011      20.703
min   1921.000       0.141
25%   1945.750       3.298
50%   1970.500      33.619
75%   1995.250      44.943
max   2020.000      65.257

Old tracks get near-zero popularity because Spotify weights recent plays.


### 5.4 `song_and_artists.csv` — Curated Songs with User Ratings

In [9]:
print(f'Shape: {song_df.shape}')
print()
print('Null values:')
print(song_df.isnull().sum().to_string())
print()
print('User-Rating format — raw values:')
print(song_df['User-Rating'].head(5).tolist())
print('=> These are strings like 9.4/10. Must parse to float before use.')
print()
print('Duplicates:', song_df.duplicated().sum())
song_df.head(3)

Shape: (2420, 5)

Null values:
Song-Name          0
Singer/Artists    10
Genre              0
Album/Movie        3
User-Rating        0

User-Rating format — raw values:
['8.8/10', '9.0/10', '9.7/10', '9.1/10', '9.2/10']
=> These are strings like 9.4/10. Must parse to float before use.

Duplicates: 16


,Song-Name,Singer/Artists,Genre,Album/Movie,User-Rating
0,Aankh Marey,"Kumar Sanu, Mika Singh, Neha Kakkar",BollywoodDance,Simmba,8.8/10
1,Coca Cola,"Neha Kakkar, Tony Kakkar",BollywoodDanceRomantic,Luka Chuppi,9.0/10
2,Apna Time Aayega,Ranveer Singh,BollywoodDance,Gully Boy,9.7/10


## 6. True Data Quality Check

Standard `isnull()` does **not** catch all data quality problems in this dataset. Below we expose the hidden issues.

In [10]:
# Issue 1: Hidden empty genres ([] strings appear as non-null but mean 'no genre')
empty_genres = (data_genre['genres'] == '[]').sum()
total        = len(data_genre)
print('=== Issue 1: Hidden empty genres in data_w_genres.csv ===')
print(f'Total artists              : {total:,}')
print(f'Null genres (pandas)       : {data_genre["genres"].isnull().sum()}')
print(f'Empty genres [] (actual)   : {empty_genres:,}  ({empty_genres/total*100:.1f}%)')
print(f'Artists WITH real genres   : {total - empty_genres:,}  ({(total-empty_genres)/total*100:.1f}%)')
print()
print('Lesson: isnull() reports 0, but 34% of artists have NO genre information.')

=== Issue 1: Hidden empty genres in data_w_genres.csv ===
Total artists              : 28,680
Null genres (pandas)       : 0
Empty genres [] (actual)   : 9,857  (34.4%)
Artists WITH real genres   : 18,823  (65.6%)

Lesson: isnull() reports 0, but 34% of artists have NO genre information.


In [11]:
# Issue 2: artists in data.csv are list strings that need parsing
print('=== Issue 2: artists column in data.csv is a Python list string ===')
raw_val = data['artists'].iloc[0]
parsed  = ast.literal_eval(raw_val)
print(f'Raw value  : {repr(raw_val)}')
print(f'After parse: {parsed}')
print(f'Type after : {type(parsed)}')
print()

# Issue 3: 2 track names are list strings
def is_list_str(s):
    s = str(s).strip()
    return s.startswith('[') and s.endswith(']')

odd_names = data[data['name'].apply(is_list_str)]
print('=== Issue 3: 2 track names stored as list strings ===')
print(odd_names[['name', 'artists', 'year']].to_string())

=== Issue 2: artists column in data.csv is a Python list string ===
Raw value  : "['Sergei Rachmaninoff', 'James Levine', 'Berliner Philharmoniker']"
After parse: ['Sergei Rachmaninoff', 'James Levine', 'Berliner Philharmoniker']
Type after : <class 'list'>

=== Issue 3: 2 track names stored as list strings ===
             name                 artists  year
37930      [oops]               ['potsu']  2017
70691  [untitled]  ['Neutral Milk Hotel']  1998


In [12]:
# Issue 4: mode in data_by_year is constant
print('=== Issue 4: mode column in data_by_year is constant ===')
print('Unique values:', year_df['mode'].unique())
print('This column should be dropped — it carries zero information.')
print()

# Issue 5: release_date mixed format
print('=== Issue 5: release_date has 3 different formats ===')
print('Sample values:', data['release_date'].sample(8, random_state=1).tolist())
print('Always use the year column for temporal analysis — it is clean and consistent.')

=== Issue 4: mode column in data_by_year is constant ===
Unique values: [1]
This column should be dropped — it carries zero information.

=== Issue 5: release_date has 3 different formats ===
Sample values: ['1990-08-07', '2020-10-30', '1966-08-30', '2020-11-11', '1983', '2003-09-22', '1970-08-31', '1995-01-01']
Always use the year column for temporal analysis — it is clean and consistent.


In [13]:
# Summary table of ALL quality issues
issues = pd.DataFrame([
    ['data_w_genres', 'genres',       'Hidden empty []',  9857, '34.3% artists have no genre — [] is not null but is empty'],
    ['data',          'artists',      'List string',      170653,'Stored as "[...]", must use ast.literal_eval() to parse'],
    ['data',          'release_date', 'Mixed format',     1610,  '3 formats: YYYY / YYYY-MM-DD / other — use year column instead'],
    ['data',          'name',         'List string (2)',  2,     '2 track names stored as ["oops"], [\'untitled\'] — edge case'],
    ['data_by_year',  'mode',         'Constant column',  100,   'Always = 1. Zero variance. Drop before modelling.'],
    ['song_and_artists','Singer/Artists','True nulls',    10,    'Standard null — fill or drop'],
    ['song_and_artists','Album/Movie', 'True nulls',      3,     'Standard null — fill or drop'],
    ['song_and_artists','User-Rating', 'String format',   2420,  'Stored as 9.4/10 — parse to float'],
], columns=['Dataset', 'Column', 'Issue Type', 'Affected Rows', 'Notes'])

issues

,Dataset,Column,Issue Type,Affected Rows,Notes
0,data_w_genres,genres,Hidden empty [],9857,34.3% artists have no genre — [] is not null b...
1,data,artists,List string,170653,"Stored as ""[...]"", must use ast.literal_eval()..."
2,data,release_date,Mixed format,1610,3 formats: YYYY / YYYY-MM-DD / other — use yea...
3,data,name,List string (2),2,"2 track names stored as [""oops""], ['untitled']..."
4,data_by_year,mode,Constant column,100,Always = 1. Zero variance. Drop before modelling.
5,song_and_artists,Singer/Artists,True nulls,10,Standard null — fill or drop
6,song_and_artists,Album/Movie,True nulls,3,Standard null — fill or drop
7,song_and_artists,User-Rating,String format,2420,Stored as 9.4/10 — parse to float


## 7. Conclusion

| # | Finding | Impact on Project |
|---|---|---|
| 1 | `genres` in `data_w_genres` has 9,857 hidden-empty `[]` values (34%) | Filter `[]` before any genre analysis |
| 2 | `artists` in `data.csv` are Python list strings | Must parse with `ast.literal_eval()` to get artist names |
| 3 | `release_date` is in 3 different formats | Use the clean `year` column for all temporal work |
| 4 | `mode` in `data_by_year` is always 1 | Drop this column it adds no information |
| 5 | `popularity` in `data_by_year` is a per-year MEAN | Very low for old tracks expected, not a data error |
| 6 | 2 track names in `data.csv` are stored as `[oops]` | Edge case parse or drop during cleaning |
| 7 | `song_and_artists` has 16 duplicates and 13 true nulls | Clean before using as evaluation data |
| 8 | `User-Rating` is a string format `9.4/10` | Parse to float in preprocessing |

**Next:** `02_eda.ipynb` explore distributions and patterns in the data.